# TP5 - Détection d’anomalie avec Python (Suite)

**Étudiant :** Maxim Khomenko

**Numéro du TP :** 5

**Date :** 20 Novembre 2026

**Version Python :** 3.13.5

**Bibliothèques utilisées :** pandas, warnings, scikit-learn, kagglehub, os, ucimlrepo

**Bibliothèques communes :**

In [1]:
import kagglehub
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score
from ucimlrepo import fetch_ucirepo

**Exercice 1 :**

In [2]:
# 1. Chargement
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
csv_path = os.path.join(path, "creditcard.csv")

df_credit = pd.read_csv(csv_path) # type: ignore

X_credit = df_credit.drop(columns=['Class', 'Time'])
y_credit = df_credit['Class']

In [3]:
# 2. Séparation des données en données d'entraînement et de test
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_credit, y_credit, test_size=0.3, stratify=y_credit, random_state=0)

In [4]:
# 3. Entraînement du modèle Isolation Forest sur les données d'entraînement
iso_credit = IsolationForest(n_estimators=100, max_samples=256, contamination=0.0017, random_state=0)
iso_credit.fit(X_train_c);

In [5]:
# 4. Prédiction des anomalies sur les données de test
pred_credit = (iso_credit.predict(X_test_c) == -1).astype(int)

In [6]:
# 5. Évaluation des performances du modèle
score_credit = -iso_credit.score_samples(X_test_c)

print("Matrice de confusion :")
print(confusion_matrix(y_test_c, pred_credit))

print("\nRapport de classification :")
print(classification_report(y_test_c, pred_credit, digits=3))

print("AUC Précision-Rappel :", round(average_precision_score(y_test_c, score_credit), 3))

Matrice de confusion :
[[85201    94]
 [  112    36]]

Rapport de classification :
              precision    recall  f1-score   support

           0      0.999     0.999     0.999     85295
           1      0.277     0.243     0.259       148

    accuracy                          0.998     85443
   macro avg      0.638     0.621     0.629     85443
weighted avg      0.997     0.998     0.998     85443

AUC Précision-Rappel : 0.154


**Interprétation :**

L'exactitude de 99,8 % est trompeuse car les fraudes ne représentent que 0,17 % des transactions. Sur les 148 fraudes du jeu de test, le modèle n'en détecte que 36 (rappel de 24,3 %) et génère 94 fausses alertes (précision de 27,7 %, F1 = 0,259), ce que confirme une AUC Précision-Rappel faible de 0,154.

**Exercice 2**

In [7]:
# 1. Chargement
spambase = fetch_ucirepo(id=94)

X_spam = spambase.data.features
y_spam = spambase.data.targets

print("Structure (lignes, colonnes) :", X_spam.shape)
print("\nDistribution des classes :\n", y_spam.value_counts())

Structure (lignes, colonnes) : (4601, 57)

Distribution des classes :
 Class
0        2788
1        1813
Name: count, dtype: int64


In [8]:
# 2. Séparation des données en données d'entraînement et de test
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_spam, y_spam, test_size=0.3, stratify=y_spam, random_state=0)

In [9]:
# 3. Entraînement du modèle Isolation Forest sur les données d'entraînement
iso_spam = IsolationForest(n_estimators=100, max_samples=256, contamination=0.394, random_state=0)
iso_spam.fit(X_train_s);

In [10]:
# 4. Prédiction des anomalies sur les données de test
pred_spam_raw = iso_spam.predict(X_test_s)
pred_spam = (pred_spam_raw == -1).astype(int)

In [11]:
# 5. Évaluation des performances du modèle
score_spam = -iso_spam.score_samples(X_test_s)

print("Matrice de confusion :")
print(confusion_matrix(y_test_s, pred_spam))

print("\nRapport de classification :")
print(classification_report(y_test_s, pred_spam, digits=3))

print("AUC Précision-Rappel :", round(average_precision_score(y_test_s, score_spam), 3))

Matrice de confusion :
[[555 282]
 [254 290]]

Rapport de classification :
              precision    recall  f1-score   support

           0      0.686     0.663     0.674       837
           1      0.507     0.533     0.520       544

    accuracy                          0.612      1381
   macro avg      0.597     0.598     0.597      1381
weighted avg      0.616     0.612     0.613      1381

AUC Précision-Rappel : 0.481


**Interprétation :**

Sur les 544 spams du jeu de test, le modèle n'en détecte que 290 (rappel de 53,3 %) et classe à tort 282 e-mails légitimes en spam (précision de 50,7 %, F1 = 0,520, AUC Précision-Rappel = 0,481), ce qui est gênant pour un filtre anti-spam où on privilégie la précision pour limiter les faux positifs. Ces résultats s'expliquent par le fait que les spams représentent 39,4 % des données, ils ne sont pas assez rares pour être isolés efficacement par Isolation Forest.